## This is the trainer for the project. 
---
We'll be using this trainer to teach the model proper cirq syntax and in turn create a model that generates accurate quantum circuits with reduced halluciations. 

In [2]:
import cirq
import torch 

print(torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print("CUDA not available or PyTorch not compiled with CUDA.")
    class _DummyProps:
        def __init__(self): self.total_memory = 0
    torch.cuda.get_device_properties = lambda idx: _DummyProps()
print(torch.cuda.get_device_properties(0).total_memory / 1e9, "GB")


False
CUDA not available or PyTorch not compiled with CUDA.
0.0 GB


In [13]:
import datasets
import peft
import trl
import transformers


from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

# Loading dataset 

dataset = load_dataset("json", data_files="dataset.json")
dataset = dataset.train_test_split(test_size=0.2)


# Model and Tokenizer 

model_name = "meta-llama/Llama-3.1-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Lora Configuring

lora_config = LoraConfig(
    r = 16, # rank - controls how many perameters we're updating, should be increased if model isnt training well
    lora_alpha=32, # scaling factor, helps develop training
    target_modules=["q_proj","v_proj"], # which layers to adapt, query and value
    lora_dropout=0.05, # helps create more robust model by focusing training by dropping some signals
    task_type="CAUSAL_LM"
    )

# Training configuring 
training_config = SFTConfig(
    output_dir = "./fine_tuned_model", # stores output
    num_training_epochs=3, #number of times model goes through training data
    per_device_train_batch_size=4, # model goes through 4 training examples at a time
    learning_rate=2e-4, # how much the model updates its parameters during training
    dataset_text_field="prompt" # what columm in dataset is used to train

)

# Da Trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    peft_config=lora_config, # perameter efficient fine tuning 
    args=training_config # passes hyperparameters/training ops 

)

trainer.train() 

Generating train split: 0 examples [00:00, ? examples/s]Failed to load JSON from file 'C:\Users\sjcap\OneDrive\infleqtion_summer\dataset.json' with error <class 'pyarrow.lib.ArrowInvalid'>: JSON parse error: Invalid value. in row 0
Generating train split: 0 examples [00:00, ? examples/s]


DatasetGenerationError: An error occurred while generating the dataset